# 📘 Tutorial 2: Prediction, Uncertainty, and Confidence

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

In **Tutorial 1**, we introduced the idea of modelling an unknown objective with a surrogate.

We saw that when evaluations are expensive:
- the objective cannot be inspected everywhere,
- only a small number of samples can be collected,
- and the surrogate becomes a practical stand-in for the unknown function.

In this tutorial, we take the next conceptual step:

> **a useful surrogate should tell us not only what it predicts, but also how uncertain that prediction is.**

This is a crucial shift.

A fitted mean curve may look smooth and convincing, but smoothness alone does not tell us:
- where the model is strongly supported by data,
- where it is extrapolating,
- or how much confidence we should place in different parts of the prediction.

Once this is recognised, modelling becomes more than curve fitting.

It becomes a problem of:
- distinguishing prediction from confidence,
- understanding how uncertainty depends on data support,
- and using both predicted value and uncertainty to guide where to evaluate next.

To make this concrete, we continue with simple one-dimensional black-box objectives and introduce a basic uncertainty band around the surrogate mean.

This lets us study not only what the model predicts, but also where it is likely to be reliable, where it is not, and how that affects optimisation decisions.

---

**This tutorial is designed to shift perspective**
- from *“a surrogate is just a fitted prediction curve”*
- to *“a useful surrogate must represent both prediction and uncertainty.”*

---

**The emphasis is on developing intuition for**
- why a mean prediction alone is incomplete,
- how uncertainty should depend on the location of observed data,
- why extrapolation should carry lower confidence than interpolation,
- how candidate points can differ in both predicted value and uncertainty,
- and how uncertainty changes the way future evaluations should be chosen.

---

**Key ideas explored include**
- the distinction between mean prediction and predictive uncertainty,
- uncertainty bands as visual representations of confidence,
- interpolation vs extrapolation,
- candidate-point comparison through $\mu(x)$ and $\sigma(x)$,
- the exploration–exploitation balance through $\mu(x) - \beta \sigma(x)$,
- and how one new observation updates both the surrogate mean and local confidence.

---

This tutorial serves as the conceptual bridge from **surrogate modelling** to **uncertainty-aware optimisation**.

In particular, it shows that once the objective is only partially observed:
- prediction alone is not enough,
- uncertainty must be modelled explicitly,
- and good decisions require balancing what the model currently believes with what it still does not know.

These ideas prepare the ground for the next tutorial, where we move from heuristic uncertainty to a more principled probabilistic framework and study **Gaussian Processes as surrogate models**.

---

**Recommended prerequisites**
- Completion of **Tutorial 1** in Part 3
- Basic familiarity with surrogate modelling and optimisation
- Comfort with simple function fitting and plots

---

**Author**: Angze Li

**Last updated**: 2026-04-14

**Version**: v2.0

## 🔧 Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)


def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)


def style_ax(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
    ax.tick_params(axis="both", labelsize=14)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight("bold")


set_seed(0)

In [ ]:
def expensive_objective(x):
    return 0.35 * torch.sin(2.2 * x) + 0.15 * torch.cos(5.0 * x) + 0.03 * (x - 1.5) ** 2 - 0.25


x_dense = torch.linspace(-3.0, 3.0, 600)
y_dense = expensive_objective(x_dense)

## 1. A prediction curve alone is not enough

In the previous tutorial, we introduced the idea of modelling an unknown objective with a surrogate.

A natural first instinct is to ask:

> if we can fit a curve to the observed data, is that already enough?

This cell begins by showing why the answer is **no**.

A surrogate mean can give us a smooth prediction of the unknown function, but a prediction curve by itself does not tell us:
- where the model is well supported by data,
- where it is making a guess,
- or how reliable different parts of the curve really are.

So while a fitted curve is useful, it is still incomplete.

---

### What the code does

The helper function `sample_objective(n_points, x_min=-3.0, x_max=3.0)` generates a small set of sampled observations from the unknown objective.

The helper function `fit_polynomial_surrogate(x_train, y_train, degree)` fits a polynomial surrogate to those observations.

The helper function `eval_polynomial_surrogate(coeffs, x)` then evaluates that fitted surrogate on a dense grid.

In this cell, we:
- sample 8 observations from the objective,
- fit a degree-5 polynomial surrogate,
- and plot the result against the true objective.

The figure therefore compares three things:
- the true objective,
- the surrogate mean,
- and the observed data points used to fit it.

---

### How to interpret the figure

The dashed surrogate curve gives a single smooth prediction across the entire domain.

At first glance, this looks useful:
- it fills in the gaps between observations,
- it gives a continuous approximation,
- and it suggests where the function may be low or high.

But this is exactly the problem:

> the curve looks equally confident everywhere, even though the data are sparse.

The model is making predictions not only near observed points, but also in regions where there is little direct evidence supporting its shape.

So the figure illustrates an important limitation:

- a prediction curve can show **what the model says**
- but it does not yet show **how much the model should be trusted**

---

### Why this matters

If we only use the surrogate mean, we might be tempted to treat every part of the fitted curve as equally informative.

But in practice, that would be misleading.

A model may be:
- quite reliable near observed data,
- much less reliable far away from those observations,
- and especially uncertain in regions where it is effectively interpolating or extrapolating with little support.

This is why a useful surrogate model must do more than just produce a mean prediction.

It must also tell us something about:

> **uncertainty, confidence, and where the prediction is weakly supported by data.**

That is the central theme of this tutorial.

---

### Key takeaway

> A fitted prediction curve is useful, but it is not enough on its own.

To make good optimisation decisions, we need not only a predicted mean, but also some way of representing **how uncertain that prediction is** across the domain.

In [ ]:
def sample_objective(n_points, x_min=-3.0, x_max=3.0):
    x_train = torch.linspace(x_min, x_max, n_points)
    y_train = expensive_objective(x_train)
    return x_train, y_train


def fit_polynomial_surrogate(x_train, y_train, degree):
    coeffs = np.polyfit(x_train.numpy(), y_train.numpy(), degree)
    return coeffs


def eval_polynomial_surrogate(coeffs, x):
    y = np.polyval(coeffs, x.numpy())
    return torch.tensor(y, dtype=x.dtype)


x_train, y_train = sample_objective(8)
coeffs = fit_polynomial_surrogate(x_train, y_train, degree=5)
y_pred = eval_polynomial_surrogate(coeffs, x_dense)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
ax.plot(x_dense.numpy(), y_pred.numpy(), linewidth=2.0, linestyle="--", label="Surrogate mean")
ax.scatter(x_train.numpy(), y_train.numpy(), s=35, zorder=3, label="Observations")
ax.set_title("A prediction curve alone", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 2. Prediction does not show where the model is trustworthy

The previous cell showed that a surrogate can produce a smooth prediction curve from a small number of observations.

This cell sharpens the problem by making the data support more explicit.

Here, the observations are collected only in the central interval

$$
[-1.5,\;1.5],
$$

while the surrogate is still plotted across the wider domain

$$
[-3,\;3].
$$

This creates a very important distinction between:
- the region where the model has actually seen data,
- and the region where it is effectively extending its prediction beyond that support.

---

### What the code does

We first sample 8 observations only from the interval

$$
x \in [-1.5,\;1.5].
$$

A degree-5 polynomial surrogate is then fitted to those observations.

The plot shows:
- the true objective over the full domain,
- the surrogate mean over the full domain,
- the observed data points,
- and a shaded band marking the interval where observations were actually collected.

So the figure is designed to contrast **where the model has evidence** with **where it is simply producing a continuation of the fitted curve**.

---

### How to interpret the figure

Inside the shaded region, the surrogate is making predictions in a part of the domain where it has direct support from observations.

Outside the shaded region, the surrogate may still look smooth and well behaved, but that smoothness should not be confused with trustworthiness.

This is the key point:

> a model can produce a numerical prediction everywhere, but that does not mean every part of the prediction is **equally reliable**.

The dashed curve does not show:
- where the model is interpolating between known points,
- where it is extrapolating beyond the data,
- or how much confidence we should place in either case.

So even though the surrogate mean is defined across the full domain, the figure already suggests that some parts of it should be trusted much less than others.

---

### Why this matters

This cell motivates one of the most important ideas in surrogate modelling:

> **prediction must be accompanied by uncertainty.**

Without uncertainty information, the surrogate mean hides a crucial distinction:
- predictions made near observed data,
- versus predictions made in regions where the model has little or no support.

This matters especially in optimisation, because a low predicted value outside the observed region may look attractive — but it may simply be a poorly supported guess.

So the lesson here is not just that the model predicts a curve.

It is that:

> **the curve alone does not tell us where the model actually knows something.**

That is exactly why the next step in the tutorial is to introduce an **uncertainty band**.

In [ ]:
x_train_mid, y_train_mid = sample_objective(8, x_min=-1.5, x_max=1.5)
coeffs_mid = fit_polynomial_surrogate(x_train_mid, y_train_mid, degree=5)
y_pred_mid = eval_polynomial_surrogate(coeffs_mid, x_dense)

fig, ax = plt.subplots(figsize=(8, 5))
ax.axvspan(-1.5, 1.5, alpha=0.12)
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
ax.plot(x_dense.numpy(), y_pred_mid.numpy(), linewidth=2.0, linestyle="--", label="Surrogate mean")
ax.scatter(x_train_mid.numpy(), y_train_mid.numpy(), s=35, zorder=3, label="Observations")
ax.set_title("Prediction does not show where the model is trustworthy", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 3. Prediction with an uncertainty band

The previous cells showed that a surrogate mean can provide a smooth prediction of the unknown objective, but that the prediction curve alone does not tell us how trustworthy different parts of that curve are.

This is why we now introduce an **uncertainty band**.

The central idea is that a useful surrogate should provide not only a predicted mean

$$
\mu(x),
$$

but also a measure of uncertainty

$$
\sigma(x),
$$

which tells us how confident or uncertain the model is at each input location.

---

### What the code does

The function `heuristic_uncertainty(x_grid, x_obs, base, scale, lengthscale)` constructs a simple uncertainty function over the input domain.

It does this by assigning each observed point a local zone of influence using Gaussian-shaped contributions. These contributions are summed and normalised to produce an overall measure of how strongly each location is supported by nearby data.

The resulting uncertainty is then defined as

$$
\sigma(x) = \text{base} + \text{scale}\,\bigl(1 - \text{influence}(x)\bigr).
$$

Here:

- $\text{influence}(x)$ measures how strongly the location $x$ is supported by nearby observations.
  It is close to **1** in regions well covered by data and closer to **0** in regions far from observed points.
- `base` sets a **minimum uncertainty floor**.
  Even in well-observed regions, the model is not forced to claim zero uncertainty.
- `scale` controls how much the uncertainty can grow when the local data support becomes weak.

So:
- where the observed data have strong local influence, $\sigma(x)$ is small,
- where the model is far from the observations, $\sigma(x)$ becomes larger.

This is not yet a full probabilistic uncertainty model, but it captures the key qualitative principle we want:

> **uncertainty should be low near data and higher far away from data.**

The code then:
- fits a polynomial surrogate mean $\mu(x)$,
- computes the uncertainty function $\sigma(x)$,
- and plots the band

$$
\mu(x) \pm 2\sigma(x).
$$

---

### What is the mathematical meaning of the uncertainty band?

The uncertainty band is the shaded region around the mean prediction:

$$
\mu(x) - 2\sigma(x)
\;\le\;
f(x)
\;\le\;
\mu(x) + 2\sigma(x).
$$

Here:
- $\mu(x)$ is the surrogate’s central prediction,
- $\sigma(x)$ represents how uncertain the model is at that point,
- and the band $\mu(x)\pm2\sigma(x)$ gives a visual range of plausible function values.

The factor of 2 is used for intuition: it creates a wider band that makes uncertainty easier to see.
At this stage, it should be interpreted qualitatively rather than as an exact probabilistic confidence interval.

So mathematically, the band is representing:

> **not just what the model predicts, but how much room there is for that prediction to be wrong.**

This is the key difference between a model with uncertainty and a model that only provides a single best-fit curve.

---

### How to interpret the figure

The figure now contains four pieces of information:

- the **true objective**
- the **mean prediction**
- the **uncertainty band**
- the **observations**

The important thing to notice is that the dashed mean prediction is no longer standing alone.

Instead, it is accompanied by a shaded region whose width changes across the domain.

- Near observed points, the band is narrower.
- In regions with less direct support, the band widens.

This visualises an important modelling principle:

> a prediction should carry different levels of confidence in different parts of the input space.

So even if two locations have similar predicted means, they may not be equally trustworthy if their uncertainty levels differ.

---

### Why this matters

This is a major conceptual step in the tutorial.

A surrogate mean alone can tell us where the function might be low, but it cannot tell us where the model is guessing.

The uncertainty band adds exactly that missing information.

This matters for optimisation because a point with:
- a low predicted mean,
- but very large uncertainty,

is very different from a point with:
- the same predicted mean,
- but small uncertainty.

So once uncertainty is introduced, optimisation decisions can no longer be based on $\mu(x)$ alone.

They must begin to account for both:
- **predicted value**
- and **predictive uncertainty**

---

### Key takeaway

> The uncertainty band represents a range of plausible values around the surrogate mean, and its width tells us how confident the model is at each point.

This is the mathematical and conceptual role of uncertainty in surrogate modelling:

- $\mu(x)$ tells us what the model predicts,
- $\sigma(x)$ tells us how uncertain that prediction is,
- and $\mu(x)\pm2\sigma(x)$ makes that uncertainty visible.

This is the foundation for the next major idea in the tutorial:

> **good optimisation decisions require not just prediction, but prediction together with uncertainty.**

In [ ]:
def heuristic_uncertainty(x_grid, x_obs, base=0.03, scale=0.35, lengthscale=0.35):
    influence = torch.zeros_like(x_grid)

    for x_i in x_obs:
        influence += torch.exp(-0.5 * ((x_grid - x_i) / lengthscale) ** 2)

    influence = influence / torch.max(influence)
    sigma = base + scale * (1.0 - influence)
    return sigma

x_train, y_train = sample_objective(8)
coeffs = fit_polynomial_surrogate(x_train, y_train, degree=5)
mu = eval_polynomial_surrogate(coeffs, x_dense)
sigma = heuristic_uncertainty(x_dense, x_train, base=0.03, scale=0.30, lengthscale=0.35)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
ax.plot(x_dense.numpy(), mu.numpy(), linewidth=2.0, linestyle="--", label="Mean prediction")
ax.fill_between(
    x_dense.numpy(),
    (mu - 2.0 * sigma).numpy(),
    (mu + 2.0 * sigma).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)
ax.scatter(x_train.numpy(), y_train.numpy(), s=35, zorder=3, label="Observations")
ax.set_title("Prediction with an uncertainty band", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 4. More observations shrink uncertainty

In the previous cell, we introduced an uncertainty band around the surrogate mean and interpreted it as a visual representation of how strongly different parts of the prediction are supported by data.

A natural next question is:

> **What happens to that uncertainty band when we collect more observations?**

This cell addresses that question directly by comparing two datasets of different sizes.

---

### What the code does

We compare two observation budgets:

- 6 observations
- 16 observations

For each case, we:

- sample the unknown objective,
- fit the same degree-5 polynomial surrogate,
- compute the same heuristic uncertainty function,
- and plot the resulting mean prediction together with the uncertainty band.

The only thing that changes between the two panels is the **number of observed data points**.

This makes it possible to isolate the effect of data density on predictive confidence.

---

### How to interpret the figure

The key visual feature to compare is the width of the uncertainty band.

With fewer observations:

- the data provide weaker support across the domain,
- the surrogate has less information to constrain its shape,
- and the uncertainty band remains relatively wide.

With more observations:

- the domain is covered more densely,
- the model has stronger local support in more places,
- and the uncertainty band becomes narrower.

So the figure illustrates a very important principle:

> **as the number of observations increases, predictive uncertainty should decrease.**

This is exactly what we want from a sensible surrogate model.

---

### The mathematical idea behind the comparison

Recall that the uncertainty function is defined as

$$
\sigma(x) = \text{base} + \text{scale}\,\bigl(1 - \text{influence}(x)\bigr),
$$

where $\text{influence}(x)$ measures how strongly the location $x$ is supported by nearby observations.

When we increase the number of observed points:

- more locations receive nearby support,
- $\text{influence}(x)$ becomes larger across more of the domain,
- and therefore $\sigma(x)$ becomes smaller.

So this comparison is not just visual — it matches the mathematical structure of the uncertainty model itself.

---

### Why this matters

This cell captures one of the most important ideas in surrogate modelling:

> **data does not only improve prediction — it also reduces uncertainty.**

That is crucial for optimisation.

If every new evaluation can reduce uncertainty in a useful region, then sampling is doing two things at once:

- improving the mean model of the objective,
- and increasing our confidence in that model.

This is one of the main reasons expensive optimisation is naturally a sequential learning problem rather than just a static fitting problem.

---

### Key takeaway

> More observations should make the surrogate not only more informed, but also more confident.

This is a foundational idea for the rest of Part 3:

- sparse data leads to wider uncertainty,
- denser data shrinks uncertainty,
- and intelligent optimisation depends on using evaluations to improve both prediction and confidence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, n in zip(axes, [6, 16]):
    x_train, y_train = sample_objective(n)
    coeffs = fit_polynomial_surrogate(x_train, y_train, degree=5)
    mu = eval_polynomial_surrogate(coeffs, x_dense)
    sigma = heuristic_uncertainty(x_dense, x_train, base=0.03, scale=0.30, lengthscale=0.35)

    ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="true objective")
    ax.plot(x_dense.numpy(), mu.numpy(), linewidth=2.0, linestyle="--", label="mean prediction")
    ax.fill_between(
        x_dense.numpy(),
        (mu - 2.0 * sigma).numpy(),
        (mu + 2.0 * sigma).numpy(),
        alpha=0.2
    )
    ax.scatter(x_train.numpy(), y_train.numpy(), s=35, zorder=3)
    ax.set_title(f"{n} observations", fontsize=18, fontweight="bold")
    ax.set_xlabel("x", fontsize=22, fontweight="bold")
    ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
    style_ax(ax)

plt.tight_layout()
plt.show()

## 5. Interpolation and extrapolation are not the same

So far, we have discussed the idea that a surrogate prediction should not be trusted equally across the whole domain.

This cell makes that idea more precise by introducing the distinction between **interpolation** and **extrapolation**.

This distinction is one of the most important ideas in surrogate modelling.

---

### What do interpolation and extrapolation mean?

Suppose the observations lie only in some interval, here

$$
[-1.2,\;1.2].
$$

Then:

- **interpolation** means making predictions **inside** the region supported by observed data,
- **extrapolation** means making predictions **outside** that region.

Mathematically, interpolation is prediction in a region where the model is constrained by nearby observations.
Extrapolation is prediction in a region where the model must continue its fitted trend without strong local data support.

That is a fundamentally harder task.

---

### What the code does

In this cell, we deliberately restrict the observations to the central interval

$$
x \in [-1.2,\;1.2].
$$

We then fit a degree-5 polynomial surrogate to those observations and plot the resulting mean prediction across the much wider domain

$$
[-3,\;3].
$$

The shaded band marks the region where the observations were actually collected.

So the figure lets us compare two kinds of prediction in one plot:

- prediction **inside** the observed region,
- prediction **outside** the observed region.

---

### How to interpret the figure

Inside the shaded interval, the surrogate is interpolating between observed data points.

This means:
- the model is constrained by nearby observations,
- the predicted shape is influenced by actual evidence,
- and the fit is generally more meaningful.

Outside the shaded interval, the surrogate is extrapolating.

This means:
- the model is no longer anchored by nearby data,
- the prediction is just a continuation of the fitted polynomial,
- and even a smooth-looking curve may be much less reliable.

This is the crucial point:

> **a model can produce a numerical prediction everywhere, but interpolation and extrapolation do not deserve the same level of trust.**

---

### Why extrapolation is especially dangerous

Extrapolation can be misleading because the surrogate must extend its learned pattern beyond the region where that pattern was actually observed.

A polynomial fitted inside $[-1.2,1.2]$ may continue very smoothly outside that interval, but that smooth continuation is not evidence that it is correct.

So even though the extrapolated prediction may *look* well behaved, it may be based on very little real information.

This is why extrapolation is especially important in optimisation:

- a region outside the observed interval may appear promising,
- but that apparent promise may come entirely from a poorly supported extrapolation,
- not from actual evidence in the data.

---

### Why this matters

This cell sets up one of the core motivations for uncertainty.

If a model only gives a mean prediction, it does not distinguish clearly between:

- **interpolation**, where prediction is relatively grounded in data,
- and **extrapolation**, where prediction is much more speculative.

That is exactly why a prediction curve alone is insufficient.

A useful surrogate should be able to reflect the fact that:

> **extrapolated predictions should usually be treated with lower confidence than interpolated ones.**

That is what the next uncertainty-band figure will make visible.

---

### Key takeaway

> Interpolation and extrapolation are not equally trustworthy.

Inside the observed region, the surrogate is constrained by data.
Outside that region, it is extending a fitted trend into unsupported territory.

This distinction is central to surrogate modelling, because optimisation decisions based on extrapolation can easily become overconfident unless uncertainty is made explicit.

In [ ]:
x_train_central, y_train_central = sample_objective(8, x_min=-1.2, x_max=1.2)
coeffs_central = fit_polynomial_surrogate(x_train_central, y_train_central, degree=3)
mu_central = eval_polynomial_surrogate(coeffs_central, x_dense)

fig, ax = plt.subplots(figsize=(8, 5))
ax.axvspan(-1.2, 1.2, alpha=0.12)
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
ax.plot(x_dense.numpy(), mu_central.numpy(), linewidth=2.0, linestyle="--", label="Mean prediction")
ax.scatter(x_train_central.numpy(), y_train_central.numpy(), s=35, zorder=3, label="Observations")
ax.set_title("Interpolation and extrapolation", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 6. Extrapolation should carry lower confidence

The previous cell introduced the distinction between **interpolation** and **extrapolation**.

We now add the uncertainty band to make that distinction more explicit.

The goal is not only to show that extrapolated predictions may be inaccurate, but also to show that they should be treated with **lower confidence** than predictions made inside the data-supported region.

---

### What the code does

We reuse the surrogate that was fitted only on the central interval

$$
[-1.2,\;1.2].
$$

The uncertainty function is then computed over the full domain, using the same heuristic principle as before:

$$
\sigma(x) = \text{base} + \text{scale}\,\bigl(1 - \text{influence}(x)\bigr).
$$

This means:

- inside the observed region, nearby data give strong support, so uncertainty remains relatively small,
- outside the observed region, local support weakens, so the uncertainty increases.

The figure then plots:

- the true objective,
- the surrogate mean prediction,
- the uncertainty band
  $$
  \mu(x)\pm 2\sigma(x),
  $$
- and the observations themselves.

The shaded interval marks the region where data were actually observed.

---

### How to interpret the figure

Inside the observed interval, the surrogate is interpolating between known points, and the uncertainty band remains relatively narrower.

Outside that interval, the model is extrapolating, and the uncertainty band widens to reflect the weaker support from data.

This is the key conceptual point:

> **predictions outside the observed region should not be trusted in the same way as predictions inside it.**

In this particular figure, the extrapolated polynomial mean becomes very large outside the data-supported region. Because of that, the uncertainty band is visually less prominent than in earlier plots.

That does **not** mean uncertainty is unimportant.
Rather, it highlights an even stronger warning:

> **an extrapolated mean can become badly behaved, and a smooth prediction outside the observed region can quickly become misleading.**

So the figure is illustrating two related ideas at once:

- extrapolation has higher uncertainty,
- and extrapolated surrogate behaviour can become unstable or unrealistic.

---

### Why this matters mathematically

The surrogate mean

$$
\mu(x)
$$

is only constrained by the observed data inside the sampled interval.

Outside that region, the fitted polynomial is no longer strongly anchored by nearby observations, so its behaviour is determined mainly by the algebraic form of the fit rather than direct evidence from data.

At the same time, the uncertainty function

$$
\sigma(x)
$$

increases because the local data support becomes weaker.

So mathematically, extrapolation is a regime where:

- the mean prediction is less reliable,
- and the uncertainty should grow.

That is exactly why extrapolation is dangerous in optimisation.

A region outside the observed interval might appear attractive if we look only at the mean prediction, but that apparent promise may come from a poorly supported or unstable extrapolation.

---

### Why this matters for optimisation

This cell reinforces an important lesson:

> **low predicted value alone is not enough if that prediction comes from extrapolation.**

In expensive optimisation, blindly trusting extrapolated minima can lead to poor or misleading decisions.

That is why uncertainty is not an optional extra.
It is part of what tells us whether a predicted optimum is genuinely promising or merely a model artefact.

---

### Key takeaway

> Extrapolation should carry lower confidence because the model is operating outside the region directly supported by data.

In this figure, the exaggerated extrapolated behaviour of the polynomial surrogate actually makes the warning even stronger:

- outside the observed region, the mean prediction can become unstable,
- and uncertainty should be treated as essential for interpreting such predictions responsibly.

In [ ]:
sigma_central = heuristic_uncertainty(x_dense, x_train_central, base=0.03, scale=0.35, lengthscale=0.30)

fig, ax = plt.subplots(figsize=(8, 5))
ax.axvspan(-1.2, 1.2, alpha=0.12)
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
ax.plot(x_dense.numpy(), mu_central.numpy(), linewidth=2.0, linestyle="--", label="Mean prediction")
ax.fill_between(
    x_dense.numpy(),
    (mu_central - 2.0 * sigma_central).numpy(),
    (mu_central + 2.0 * sigma_central).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)
ax.scatter(x_train_central.numpy(), y_train_central.numpy(), s=35, zorder=3, label="Observations")
ax.set_title("Extrapolation should carry lower confidence", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 7. Uncertainty grows outside the observed region

The previous figure combined the surrogate mean and the uncertainty band in a single plot.

To make the uncertainty structure easier to see on its own, this cell now plots only

$$
\sigma(x),
$$

the uncertainty function across the input domain.

This isolates the behaviour of the uncertainty term and makes the interpolation–extrapolation distinction more explicit.

---

### What the code does

The uncertainty function `sigma_central` was computed using observations that lie only in the central interval

$$
[-1.2,\;1.2].
$$

The plot then shows:

- the uncertainty value $\sigma(x)$ across the full domain,
- together with a shaded band marking the region where observations were actually collected.

So this figure is designed to answer a simple question directly:

> **how does model uncertainty vary with distance from the observed data?**

---

### How to interpret the figure

Inside the shaded interval, the uncertainty remains relatively low because the model has nearby observations supporting its prediction.

Outside that interval, the uncertainty increases because the data support becomes weaker.

This is exactly the intended behaviour of the uncertainty model:

- **interpolation region** → lower uncertainty
- **extrapolation region** → higher uncertainty

So the figure makes visually clear that uncertainty is not uniform across the domain.
It depends on where the model has evidence and where it does not.

---

### Why this matters

This plot captures one of the most important ideas in surrogate modelling:

> **confidence should depend on data support.**

A surrogate should not treat all parts of the input space equally.
Where observations are available, prediction can be more confident.
Where the model is forced to extrapolate, confidence should fall.

This is why uncertainty is essential in expensive optimisation:
it helps distinguish between regions that are well understood and regions where the model is still largely guessing.

---

### Key takeaway

> Uncertainty should grow outside the observed region because extrapolated predictions are less strongly supported by data.

This is the conceptual role of $\sigma(x)$:
it tells us not just where the model predicts something, but where that prediction should be treated cautiously.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.axvspan(-1.2, 1.2, alpha=0.12)
ax.plot(x_dense.numpy(), sigma_central.numpy(), linewidth=2.5)
ax.set_title("Uncertainty grows outside the observed region", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel(r"$\sigma(x)$", fontsize=22, fontweight="bold")
style_ax(ax)
plt.show()

## 8. Candidate points can differ in predicted value and uncertainty

Up to this point, the tutorial has built a crucial distinction:

- the surrogate mean tells us **what the model predicts**
- the uncertainty function tells us **how strongly that prediction is supported**

The next step is to use these two quantities together.

This cell shows that different candidate points can look attractive for different reasons, and that a sensible optimisation strategy should not judge candidate points by predicted mean alone.

---

### What the code does

We first fit a surrogate model using 8 observations from the central interval

$$
[-1.5,\;1.5].
$$

From that surrogate, we compute:

- the mean prediction
  $$
  \mu(x),
  $$
- and the uncertainty estimate
  $$
  \sigma(x).
  $$

We then choose three candidate points:

$$
x \in \{-2.5,\;0,\;1.8\},
$$

and evaluate both:
- the surrogate mean at those locations,
- and the corresponding uncertainty.

The figure shows:
- the mean prediction curve,
- the uncertainty band
  $$
  \mu(x)\pm 2\sigma(x),
  $$
- the observed data,
- and the candidate points themselves.

Each candidate is also annotated with its numerical values of $\mu$ and $\sigma$.

---

### How to interpret the figure

The main lesson is that candidate points can differ in two distinct ways:

- they may have different **predicted values**
- and they may have different **uncertainty levels**

This matters because a point with a low predicted mean is not automatically the best next choice if that prediction is very weakly supported.

Likewise, a point with high uncertainty is not automatically attractive if its predicted value is clearly poor.

The three candidates in the figure are meant to illustrate different roles:

- one may look very promising under the mean prediction, but also carry high uncertainty,
- one may be well supported by existing data, but not especially exciting,
- one may lie somewhere in between, combining a moderately good mean with moderate uncertainty.

So even before defining a formal acquisition rule, this figure already makes an important point:

> **good candidate selection depends on both prediction and uncertainty.**

---

### The mathematical idea behind the comparison

Each candidate point is associated with a pair

$$
\bigl(\mu(x),\sigma(x)\bigr).
$$

This pair contains two different kinds of information:

- $\mu(x$ measures how good the point currently looks according to the model
- $\sigma(x)$ measures how uncertain the model is about that assessment

A point with:
- low $\mu(x)$ and low $\sigma(x)$

looks like a strong **exploitation** candidate: the model believes it is good and is relatively confident.

A point with:
- moderate or even mediocre $\mu(x)$, but high $\sigma(x)$

may still be worth evaluating as an **exploration** candidate, because the true objective there could turn out to be much better than the current mean suggests.

This is the first place in the tutorial where the exploration–exploitation tension becomes mathematically visible.

---

### Why this matters for optimisation

If we were to choose the next evaluation point using only the mean prediction, we would always prefer the point with the lowest $\mu(x)$.

But that strategy ignores the possibility that a more uncertain point could reveal something better.

Conversely, if we only chase uncertainty, we may waste evaluations in regions that are uncertain but not promising.

So the real optimisation problem is not:

> “which point has the lowest predicted value?”

but rather:

> “which point best balances predicted value and uncertainty?”

That is exactly the question that leads to the next step of the tutorial.

---

### Key takeaway

> Candidate points should not be judged by predicted value alone.

A useful surrogate-based decision must consider both:
- how good a point currently looks,
- and how uncertain that assessment is.

This is the conceptual bridge to the next idea:

> **we need a rule that combines $\mu(x)$ and $\sigma(x)$ into a single score for choosing the next evaluation.**

In [ ]:
x_train, y_train = sample_objective(8, x_min=-1.5, x_max=1.5)
coeffs = fit_polynomial_surrogate(x_train, y_train, degree=5)
mu = eval_polynomial_surrogate(coeffs, x_dense)
sigma = heuristic_uncertainty(x_dense, x_train, base=0.03, scale=0.30, lengthscale=0.35)

candidate_points = torch.tensor([-2.5, 0, 1.8])
mu_candidates = eval_polynomial_surrogate(coeffs, candidate_points)
sigma_candidates = heuristic_uncertainty(candidate_points, x_train, base=0.03, scale=0.30, lengthscale=0.35)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), mu.numpy(), linewidth=2.0, linestyle="--", label="Mean prediction")
ax.fill_between(
    x_dense.numpy(),
    (mu - 2.0 * sigma).numpy(),
    (mu + 2.0 * sigma).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)
ax.scatter(x_train.numpy(), y_train.numpy(), s=35, zorder=3, label="Observations")
ax.scatter(candidate_points.numpy(), mu_candidates.numpy(), s=70, zorder=4, label="Candidate points")

for x_c, mu_c, sig_c in zip(candidate_points, mu_candidates, sigma_candidates):
    ax.text(float(x_c), float(mu_c) + 0.8, f"μ={float(mu_c):.2f}\nσ={float(sig_c):.2f}",
            fontsize=10, fontweight="bold", ha="center")

ax.set_title("Candidate points can differ in value and uncertainty", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("Prediction", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 9. A simple combined score for balancing prediction and uncertainty

The previous cell showed that candidate points can differ not only in predicted value, but also in uncertainty.

That immediately raises a practical question:

> **How should we combine these two quantities when choosing the next evaluation point?**

This cell introduces a simple heuristic score that balances:
- **exploitation**: preferring points with low predicted mean
- **exploration**: preferring points with high uncertainty

---

### What the code does

We define the score

$$
s(x) = \mu(x) - \beta \sigma(x),
$$

where:
- $\mu(x)$ is the surrogate mean prediction,
- $\sigma(x)$ is the uncertainty estimate,
- and $\beta \ge 0$ controls how strongly uncertainty is rewarded.

For a minimisation problem:

- smaller $\mu(x)$ makes a point look attractive because the model predicts a low objective value there,
- larger $\sigma(x)$ also makes a point attractive because the true objective could turn out to be better than the current mean suggests.

So the term $-\beta \sigma(x)$ introduces an explicit incentive to explore uncertain regions.

The code then:
- fits the surrogate mean and uncertainty as before,
- restricts the candidate search to a meaningful interval,
- normalises the mean and uncertainty terms,
- and evaluates the combined score for three values of $\beta$:

$$
\beta \in \{0.0,\;0.5,\;1.0\}.
$$

For each value of $\beta$, the point with the smallest score is selected and marked on the plot.

---

### Why are the mean and uncertainty normalised?

The raw surrogate mean and uncertainty often live on very different numerical scales.

If we combine them directly, the score may be dominated by whichever term happens to be numerically larger, rather than reflecting the intended conceptual trade-off.

To make the role of $\beta$ easier to see, the code standardises both quantities over the search region:

- the mean prediction is centred and scaled,
- the uncertainty is also centred and scaled.

This makes the combined score more interpretable:

> changing $\beta$ now genuinely changes the balance between predicted value and uncertainty, rather than just being overwhelmed by a scale mismatch.

So the score being plotted is best understood as a **normalised exploration–exploitation score**, not as a literal acquisition function yet.

---

### How to interpret the three panels

Each panel shows the score function for a different value of $\beta$.

#### $\beta = 0$
This gives

$$
s(x) = \mu(x),
$$

so the model is choosing purely by predicted mean.

This is **pure exploitation**:
- select the point that currently looks best,
- ignore uncertainty completely.

#### $\beta = 0.5$
Now uncertainty begins to matter.

The selected point is influenced by both:
- how low the predicted mean is,
- and how uncertain that prediction remains.

This is a **mixed exploration–exploitation regime**.

#### $\beta = 1.0$
Uncertainty is weighted even more strongly.

Now the selected point may move toward regions that are not the lowest in mean prediction, but are uncertain enough to deserve investigation.

This reflects a stronger preference for **exploration**.

---

### The mathematical argument behind the score

The reason this score is useful is that it turns two different modelling quantities into one decision quantity.

The mean prediction $\mu(x)$ answers:

> **Where does the model currently think the objective is low?**

The uncertainty $\sigma(x)$ answers:

> **Where could the model still be wrong in an interesting way?**

The combined score

$$
\mu(x) - \beta \sigma(x)
$$

therefore encourages points that are:
- low in predicted value,
- high in uncertainty,
- or both.

So mathematically, $\beta$ controls the trade-off:
- smaller $\beta$ → more exploitation
- larger $\beta$ → more exploration

This is the core intuition that later becomes formalised in Bayesian Optimisation.

---

### Why this matters

This cell is the first place in the tutorial where uncertainty becomes part of a **decision rule**, rather than just part of the visualisation.

That is an important conceptual transition.

We are no longer asking only:

- what does the model predict?
- how uncertain is it?

We are now asking:

> **how should prediction and uncertainty be combined to choose the next evaluation?**

That is the key question behind model-guided search.

---

### Key takeaway

> A useful surrogate should not only predict the objective and quantify uncertainty — it should support a rule for balancing the two when deciding where to sample next.

The score

$$
\mu(x) - \beta \sigma(x)
$$

is a simple first example of that idea.

It is not yet a full Bayesian Optimisation acquisition function, but it already captures the essential principle:

- low predicted value encourages exploitation,
- high uncertainty encourages exploration,
- and $\beta$ determines the balance between them.

In [ ]:
def optimistic_score(mu, sigma, beta=1.0):
    return mu - beta * sigma

x_train, y_train = sample_objective(8, x_min=-1.5, x_max=1.5)
coeffs = fit_polynomial_surrogate(x_train, y_train, degree=5)
mu = eval_polynomial_surrogate(coeffs, x_dense)
sigma = heuristic_uncertainty(x_dense, x_train, base=0.03, scale=0.30, lengthscale=0.35)

candidate_mask = (x_dense >= -2.2) & (x_dense <= 2.2)
x_search = x_dense[candidate_mask]
mu_search = mu[candidate_mask]
sigma_search = sigma[candidate_mask]

# Normalise mean and uncertainty so beta visibly changes the trade-off
mu_search_norm = (mu_search - torch.mean(mu_search)) / torch.std(mu_search)
sigma_search_norm = (sigma_search - torch.mean(sigma_search)) / torch.std(sigma_search)

betas = [0.0, 0.5, 1.0]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, beta in zip(axes, betas):
    score = optimistic_score(mu_search_norm, sigma_search_norm, beta=beta)
    idx = torch.argmin(score)
    x_next = x_search[idx]
    score_next = score[idx]

    ax.plot(x_search.numpy(), score.numpy(), linewidth=2.0)
    ax.scatter([float(x_next)], [float(score_next)], s=90, marker="*", zorder=4)
    ax.axvline(float(x_next), linewidth=1.8, linestyle="--", alpha=0.7)
    ax.set_title(rf"$\beta={beta}$", fontsize=18, fontweight="bold")
    ax.set_xlabel("x", fontsize=22, fontweight="bold")
    ax.set_ylabel("Normalised score", fontsize=22, fontweight="bold")
    style_ax(ax)

plt.tight_layout()
plt.show()

## 10. Changing $\beta$ changes the exploration–exploitation balance

The previous cell showed how the combined score

$$
s(x) = \mu(x) - \beta \sigma(x)
$$

changes as $\beta$ varies.

That score plot is useful, but the more important question is:

> **How does changing $\beta$ affect the actual point chosen for the next evaluation?**

This cell answers that question directly by placing the selected point back onto the original prediction-and-uncertainty plot.

---

### What the code does

For each value of $\beta$,

$$
\beta \in \{0.0,\;0.5,\;1.0\},
$$

the code:

- computes the normalised exploration–exploitation score on the search region,
- finds the point with the smallest score,
- maps that selected point back to the original surrogate mean plot,
- and marks it with a star and a vertical dashed line.

The shaded vertical region indicates the interval over which the next-point search is allowed. This prevents the decision rule from being dominated by extreme polynomial behaviour far outside the meaningful search region.

So each panel shows the same surrogate mean and uncertainty band, but a different selected point depending on how strongly uncertainty is rewarded.

---

### How to interpret the three panels

This figure makes the role of $\beta$ much more concrete.

#### $\beta = 0$
The selected point is chosen purely by the mean prediction.

This is **pure exploitation**:
- trust the current model,
- go where the predicted value is lowest,
- ignore uncertainty entirely.

#### $\beta = 0.5$
The selected point shifts because uncertainty now contributes to the score.

This is a more balanced regime:
- low predicted value still matters,
- but uncertainty is now strong enough to influence the choice.

#### $\beta = 1.0$
The selected point shifts further toward a more uncertain region.

This is a more exploratory regime:
- the method becomes more willing to investigate places that are less well understood,
- even if they are not the single lowest points under the mean prediction alone.

So the sequence of panels shows exactly what $\beta$ does:

> **it changes how boldly the method is willing to explore.**

---

### The exploration–exploitation balance

This is the central idea of the cell.

In expensive optimisation, there are always two competing goals:

- **exploitation**: evaluate where the surrogate already predicts a low objective value
- **exploration**: evaluate where the surrogate is uncertain and could still be missing something important

These goals are often in tension.

If we exploit too aggressively:
- we may keep sampling points that look good under the current model,
- but miss regions where the model is uncertain and wrong.

If we explore too aggressively:
- we may spend too many evaluations on uncertain regions,
- without taking advantage of what the model already knows.

The parameter $\beta$ acts as the balance knob between these two tendencies.

Mathematically:

- smaller $\beta$ means the score is dominated by $\mu(x)$,
- larger $\beta$ means the uncertainty term $\sigma(x)$ has more influence.

So this cell is visualising the exploration–exploitation trade-off in perhaps its simplest form.

---

### Why this matters

This is an important transition point in the tutorial.

Up to now, uncertainty has been introduced mainly as a way of interpreting predictions.
Here, uncertainty becomes part of the actual **decision rule**.

That is the key step from:
- passive modelling
to
- active sequential optimisation.

The method is no longer just asking:

- what is the predicted objective value?
- how uncertain is that prediction?

It is now asking:

> **how should those two quantities be combined to decide where to evaluate next?**

That is the heart of model-guided optimisation.

---

### Key takeaway

> Changing $\beta$ changes the exploration–exploitation balance, and therefore changes which point is selected next.

This is why uncertainty matters so much in optimisation:
it does not just tell us how much we know — it changes how we decide.

In the next section, we will see one more important consequence of this idea:

> once a new point is evaluated, the model should update not only its mean prediction, but also its confidence in the surrounding region.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, beta in zip(axes, betas):
    score = optimistic_score(mu_search_norm, sigma_search_norm, beta=beta)
    idx = torch.argmin(score)
    x_next = x_search[idx]
    mu_next = mu_search[idx]

    ax.plot(x_dense.numpy(), mu.numpy(), linewidth=2.0, linestyle="--", label="Mean prediction")
    ax.fill_between(
        x_dense.numpy(),
        (mu - 2.0 * sigma).numpy(),
        (mu + 2.0 * sigma).numpy(),
        alpha=0.2,
        label=r"$\mu(x)\pm 2\sigma(x)$"
    )
    ax.axvspan(float(x_search[0]), float(x_search[-1]), alpha=0.08)
    ax.scatter(x_train.numpy(), y_train.numpy(), s=35, zorder=3, label="Observations")
    ax.scatter([float(x_next)], [float(mu_next)], s=90, marker="*", zorder=4, label="Selected point")
    ax.axvline(float(x_next), linewidth=1.8, linestyle="--", alpha=0.7)
    ax.set_title(rf"Selected point with $\beta={beta}$", fontsize=16, fontweight="bold")
    ax.set_xlabel("x", fontsize=18, fontweight="bold")
    ax.set_ylabel("Prediction", fontsize=18, fontweight="bold")
    style_ax(ax)

axes[0].legend(prop={"size": 10, "weight": "bold"})
plt.tight_layout()
plt.show()

## 11. Adding one new observation updates both prediction and confidence

So far, the tutorial has shown that uncertainty can be used to guide the choice of a new evaluation point.

This cell now shows what happens **after** that new point is actually evaluated and added to the dataset.

This is an important step, because surrogate-based optimisation is not a one-shot procedure.

It is a **sequential learning process**:

- fit a model,
- choose a promising next point,
- evaluate the true objective there,
- update the dataset,
- and refit the model.

The purpose of this cell is to make that update visually explicit.

---

### What the code does

We begin with 8 observations in the interval

$$
[-1.5,\;1.5].
$$

From those observations, we compute:

- the surrogate mean before the new evaluation,
  $$
  \mu_{\mathrm{before}}(x),
  $$
- and the uncertainty before the new evaluation,
  $$
  \sigma_{\mathrm{before}}(x).
  $$

We then restrict the candidate search to the interval

$$
[-2.2,\;2.2],
$$

normalise the mean and uncertainty terms, and use the exploration–exploitation score

$$
s(x) = \mu(x) - \beta \sigma(x)
$$

with

$$
\beta = 1.0
$$

to choose a new point.

That new point is evaluated on the true objective and appended to the dataset.

The model is then refit using the updated observations, giving:

- the updated surrogate mean,
  $$
  \mu_{\mathrm{after}}(x),
  $$
- and the updated uncertainty,
  $$
  \sigma_{\mathrm{after}}(x).
  $$

The two panels compare the model **before** and **after** the new observation is added.

---

### How to interpret the figure

The left panel shows the model before the new observation is incorporated.

It includes:
- the mean prediction,
- the uncertainty band
  $$
  \mu_{\mathrm{before}}(x)\pm 2\sigma_{\mathrm{before}}(x),
  $$
- the existing observations,
- and the newly selected point marked with a star.

The right panel shows the model after that point has been evaluated and included in the dataset.

It includes:
- the updated mean prediction,
- the updated uncertainty band
  $$
  \mu_{\mathrm{after}}(x)\pm 2\sigma_{\mathrm{after}}(x),
  $$
- and the updated observation set.

So the figure is showing how one additional observation changes the surrogate itself.

---

### What is the main mathematical idea?

The important point is that a new evaluation changes **two things at once**:

1. the predicted mean,
   $$
   \mu(x),
   $$
2. the uncertainty,
   $$
   \sigma(x).
   $$

A new observation gives the model more information about the function in the neighbourhood of that point.

So after the update, we expect:

- the mean prediction to adjust locally,
- and the uncertainty to decrease near the newly observed location.

That is the core mechanism of sequential surrogate-based optimisation:

> each evaluation improves not only what the model predicts, but also how confident it is in that prediction.

---

### Why this matters

This is one of the most important ideas in the tutorial.

A surrogate model is not static.

Its job is not just to fit the current data once, but to be repeatedly updated as new information arrives.

That means every new evaluation has two roles:

- it can improve the estimate of the objective,
- and it can reduce uncertainty in a strategically useful part of the domain.

This is why expensive optimisation is naturally a **learning loop** rather than a fixed optimisation problem.

The model changes, the confidence changes, and therefore the next best point to evaluate may also change.

---

### Key takeaway

> Adding one new observation updates both the surrogate mean and the model’s confidence.

This is the essence of sequential model-guided optimisation:

- choose a point using the current model,
- evaluate the true objective there,
- update the model,
- and repeat.

The next cell will make one part of this update even clearer by isolating the uncertainty itself and showing how it changes before and after the new evaluation.

In [ ]:
x_train, y_train = sample_objective(8, x_min=-1.5, x_max=1.5)
coeffs = fit_polynomial_surrogate(x_train, y_train, degree=5)
mu_before = eval_polynomial_surrogate(coeffs, x_dense)
sigma_before = heuristic_uncertainty(x_dense, x_train, base=0.03, scale=0.30, lengthscale=0.35)

candidate_mask = (x_dense >= -2.2) & (x_dense <= 2.2)
x_search = x_dense[candidate_mask]
mu_search = mu_before[candidate_mask]
sigma_search = sigma_before[candidate_mask]

mu_search_norm = (mu_search - torch.mean(mu_search)) / torch.std(mu_search)
sigma_search_norm = (sigma_search - torch.mean(sigma_search)) / torch.std(sigma_search)

score = optimistic_score(mu_search_norm, sigma_search_norm, beta=1.0)
x_new = x_search[torch.argmin(score)].unsqueeze(0)
y_new = expensive_objective(x_new)

x_train_updated = torch.cat([x_train, x_new])
y_train_updated = torch.cat([y_train, y_new])

coeffs_updated = fit_polynomial_surrogate(x_train_updated, y_train_updated, degree=5)
mu_after = eval_polynomial_surrogate(coeffs_updated, x_dense)
sigma_after = heuristic_uncertainty(x_dense, x_train_updated, base=0.03, scale=0.30, lengthscale=0.35)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].plot(x_dense.numpy(), mu_before.numpy(), linewidth=2.0, linestyle="--", label="Mean prediction")
axes[0].fill_between(
    x_dense.numpy(),
    (mu_before - 2.0 * sigma_before).numpy(),
    (mu_before + 2.0 * sigma_before).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)
axes[0].scatter(x_train.numpy(), y_train.numpy(), s=35, zorder=3, label="Observations")
axes[0].scatter(x_new.numpy(), y_new.numpy(), s=90, marker="*", zorder=4, label="New point")
axes[0].set_title("Before adding new observation", fontsize=18, fontweight="bold")
axes[0].set_xlabel("x", fontsize=22, fontweight="bold")
axes[0].set_ylabel("Prediction", fontsize=22, fontweight="bold")
axes[0].legend(prop={"size": 10, "weight": "bold"}, loc="upper left")
axes[0].set_ylim(-10, 12)
style_ax(axes[0])

axes[1].plot(x_dense.numpy(), mu_after.numpy(), linewidth=2.0, linestyle="--", label="Mean prediction")
axes[1].fill_between(
    x_dense.numpy(),
    (mu_after - 2.0 * sigma_after).numpy(),
    (mu_after + 2.0 * sigma_after).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)
axes[1].scatter(x_train_updated.numpy(), y_train_updated.numpy(), s=35, zorder=3, label="Updated observations")
axes[1].set_title("After adding new observation", fontsize=18, fontweight="bold")
axes[1].set_xlabel("x", fontsize=22, fontweight="bold")
axes[1].set_ylabel("Prediction", fontsize=22, fontweight="bold")
axes[1].legend(prop={"size": 10, "weight": "bold"}, loc="upper left")
axes[1].set_ylim(-10, 12)
style_ax(axes[1])

plt.tight_layout()
plt.show()

## 12. A new evaluation reduces local uncertainty

The previous figure showed that adding a new observation updates both the surrogate mean and the uncertainty band.

This cell focuses on the uncertainty update alone.

By plotting the uncertainty function before and after the new evaluation, we can see more clearly how one additional observation changes the model’s confidence across the input domain.

---

### What the code does

Using the same sequential update from the previous cell, we compare:

- the uncertainty before the new point is observed,
  $$
  \sigma_{\mathrm{before}}(x),
  $$
- and the uncertainty after that point is added to the dataset,
  $$
  \sigma_{\mathrm{after}}(x).
  $$

The selected new input location is marked by a vertical dashed line.

So this figure isolates one very specific question:

> **How does model uncertainty change when a new evaluation is made at a particular location?**

---

### How to interpret the figure

The key feature to look for is the region near the new observation.

After the new point is added:

- the uncertainty curve drops locally around that location,
- while regions far away change much less.

This is exactly the behaviour we would want from a sensible uncertainty model.

A new evaluation should provide new information primarily in its local neighbourhood, not uniformly across the whole domain.

So the figure shows a very important structural property of sequential learning:

> **new data reduces uncertainty most strongly near where it is acquired.**

---

### The mathematical idea behind the update

The uncertainty function depends on how strongly each location is supported by nearby observations.

Before the new point is added, the support in its neighbourhood is weaker, so uncertainty is higher.

After the new point is added, that local support increases, which makes the influence term larger and therefore reduces

$$
\sigma(x) = \text{base} + \text{scale}\,\bigl(1 - \text{influence}(x)\bigr).
$$

So mathematically, the drop in uncertainty is a direct consequence of adding an observation that increases local data support.

This is why the effect is not global and uniform.
It is strongest near the newly sampled location.

---

### Why this matters

This cell makes one of the most important ideas in surrogate-based optimisation especially clear:

> **evaluations are valuable not only because they reveal function values, but also because they reduce uncertainty.**

That is what makes sequential sampling strategic.

When deciding where to evaluate next, we are not only choosing where to improve the current estimate of the objective.
We are also choosing where to increase confidence and reduce ignorance.

So a new point has two effects:
- it updates the mean prediction,
- and it sharpens the model’s local confidence structure.

---

### Key takeaway

> A new evaluation should reduce uncertainty primarily in the region where the new information was obtained.

This is a central reason surrogate modelling is useful in expensive optimisation:

- every evaluation teaches the model something new,
- and that new information changes both what the model predicts and how confident it is.

With this, the main conceptual arc of the tutorial is complete:

- prediction alone is not enough,
- uncertainty must be modelled explicitly,
- extrapolation should carry lower confidence,
- decisions should balance predicted value and uncertainty,
- and new observations should update both the surrogate mean and the model’s confidence.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), sigma_before.numpy(), linewidth=2.5, label="Before")
ax.plot(x_dense.numpy(), sigma_after.numpy(), linewidth=2.5, label="After")
ax.axvline(float(x_new), linewidth=2.0, linestyle="--", color="green")
ax.set_title("A new evaluation reduces local uncertainty", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel(r"$\sigma(x)$", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 🧭 Closing Remarks

In this tutorial, we moved beyond surrogate modelling as a way of fitting a curve and introduced a more important idea:

> **a useful surrogate should express not only what it predicts, but also how uncertain that prediction is.**

That shift is essential.

In the previous tutorial, we saw that a surrogate could approximate an unknown objective and help guide search.
Here, we learned that prediction alone is incomplete.
A smooth mean curve may look convincing, but without uncertainty, it cannot tell us:
- where the model is strongly supported by data,
- where it is extrapolating,
- or how much confidence we should place in different parts of the prediction.

---

By introducing the pair

$$
\mu(x), \qquad \sigma(x),
$$

we gave the surrogate a richer interpretation.

The mean prediction
$$
\mu(x)
$$
tells us what the model currently believes about the objective.

The uncertainty
$$
\sigma(x)
$$
tells us how strongly that belief is supported.

Together, they allow the model to represent not just a best guess, but also its **degree of confidence** in that guess.

That is the key conceptual step of the tutorial.

---

A clear pattern emerged throughout the notebook:

- near observed data, uncertainty should be lower,
- far from observed data, uncertainty should be higher,
- and extrapolation should therefore carry lower confidence than interpolation.

This gave mathematical meaning to the uncertainty band

$$
\mu(x) \pm 2\sigma(x),
$$

which is not just a visual decoration, but a way of expressing how much room there is for the prediction to be wrong.

So one of the main lessons of the tutorial is:

> **a prediction should never be interpreted independently of the uncertainty attached to it.**

---

We then connected uncertainty to optimisation decisions.

By comparing candidate points through both
$$
\mu(x)
$$
and
$$
\sigma(x),
$$
we saw that promising points can differ in two important ways:
- some look good because the current mean is low,
- others are worth investigating because uncertainty is high.

This led naturally to the combined score

$$
\mu(x) - \beta \sigma(x),
$$

where the parameter
$$
\beta
$$
controls the exploration–exploitation balance.

That was an important transition point in the tutorial, because uncertainty stopped being just something we visualise and became something that actively changes **what point is chosen next**.

---

The final sections showed that once a new observation is added, the surrogate changes in two coupled ways:

- the mean prediction is updated,
- and the uncertainty is reduced locally near the new point.

This makes clear that surrogate-based optimisation is a **sequential learning process**.

Each evaluation does not simply reveal one more function value.
It changes the model’s understanding of the objective and reshapes where future evaluations should go.

So the deeper lesson is this:

> **in expensive optimisation, evaluations are valuable not only because they improve the estimate of the objective, but also because they reduce uncertainty in strategically important regions.**

---

If there is one central takeaway from this tutorial, it is the following:

> **Prediction is not enough. Good optimisation decisions require prediction together with uncertainty.**

That is the conceptual bridge to the next tutorial.

In **Tutorial 3**, we will move from heuristic uncertainty to a more principled probabilistic framework and study:

> **Gaussian Processes as surrogate models that naturally provide both a predictive mean and a predictive uncertainty.**

That is the next major step toward full **Bayesian Optimisation**.